# Laboratório Prático 1 - Dashboard Analítico de Vendas Globais

Pergunta 1 - Qual o valor total vendido?

Pergunta 2 - Quantas vendas foram realizadas por categoria de produto?

Pergunta 3 - Quantas vendas foram realizadas por país considerando a prioridade de entrega?

Pergunta 4 - Qual foi a média de desconto nas vendas por subcategoria de produto?

Pergunta 5 - Quais países tiveram maior média de valor de venda? Demonstre em um mapa.

E nosso Dashboard deve dar ao usuário a possibilidade de filtrar os dados por ano, por segmento e por país.


## Importando as bibliotecas

In [1]:
#install
#!pip install -q -U matplotlib numpy pandas plotly scikit-learn scipy seaborn shap statsmodels xgboost

In [1]:
#imports
import pandas as pd
import numpy as np

## Acessando os dados

In [2]:
dataset = pd.read_csv('15-dataset/dataset_limpo.csv')

In [3]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 51281 entries, 0 to 51280
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ID_Pedido     51281 non-null  str    
 1   Data_Pedido   51281 non-null  str    
 2   ID_Cliente    51281 non-null  str    
 3   Segmento      51281 non-null  str    
 4   Regiao        51281 non-null  str    
 5   Pais          51281 non-null  str    
 6   Product ID    51281 non-null  str    
 7   Categoria     51281 non-null  str    
 8   SubCategoria  51281 non-null  str    
 9   Total_Vendas  51281 non-null  float64
 10  Quantidade    51281 non-null  int64  
 11  Desconto      51281 non-null  float64
 12  Lucro         51281 non-null  float64
 13  Prioridade    51281 non-null  str    
dtypes: float64(3), int64(1), str(10)
memory usage: 5.5 MB


In [4]:
dataset.head(1)

,ID_Pedido,Data_Pedido,ID_Cliente,Segmento,Regiao,Pais,Product ID,Categoria,SubCategoria,Total_Vendas,Quantidade,Desconto,Lucro,Prioridade
0,CA-2012-124891,2012-07-31,RH-19495,Consumidor,New York,United States,TEC-AC-10003033,Tecnologia,Accessories,2309.65,7,0.0,762.1845,Critico


### Correção do data_pedido, não está em typo Data

In [5]:
dataset['Data_Pedido'] = pd.to_datetime(dataset['Data_Pedido'], errors= 'coerce', format="%Y-%m-%d")

In [6]:
dataset.head(1)

,ID_Pedido,Data_Pedido,ID_Cliente,Segmento,Regiao,Pais,Product ID,Categoria,SubCategoria,Total_Vendas,Quantidade,Desconto,Lucro,Prioridade
0,CA-2012-124891,2012-07-31,RH-19495,Consumidor,New York,United States,TEC-AC-10003033,Tecnologia,Accessories,2309.65,7,0.0,762.1845,Critico


In [7]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 51281 entries, 0 to 51280
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   ID_Pedido     51281 non-null  str           
 1   Data_Pedido   51281 non-null  datetime64[us]
 2   ID_Cliente    51281 non-null  str           
 3   Segmento      51281 non-null  str           
 4   Regiao        51281 non-null  str           
 5   Pais          51281 non-null  str           
 6   Product ID    51281 non-null  str           
 7   Categoria     51281 non-null  str           
 8   SubCategoria  51281 non-null  str           
 9   Total_Vendas  51281 non-null  float64       
 10  Quantidade    51281 non-null  int64         
 11  Desconto      51281 non-null  float64       
 12  Lucro         51281 non-null  float64       
 13  Prioridade    51281 non-null  str           
dtypes: datetime64[us](1), float64(3), int64(1), str(9)
memory usage: 5.5 MB


Pergunta 1 - Qual o valor total vendido?

In [8]:
total_vendido = dataset['Total_Vendas'].sum()
total_vendido

np.float64(12641024.34188)

Pergunta 2 - Quantas vendas foram realizadas por categoria de produto?

In [9]:
categorias_unicas = dataset['Categoria'].unique()
categorias_unicas

<StringArray>
['Tecnologia', 'Moveis', 'Suprimentos']
Length: 3, dtype: str

In [10]:
contagem_produtos_por_categorias = dataset['Categoria'].value_counts().reset_index()
contagem_produtos_por_categorias.columns = ['Categoria', 'Contagem']
contagem_produtos_por_categorias

,Categoria,Contagem
0,Suprimentos,31268
1,Tecnologia,10140
2,Moveis,9873


Pergunta 3 - Quantas vendas foram realizadas por país considerando a prioridade de entrega?

In [11]:
#groupby
paises_vendas_por_prioridade = dataset.groupby(['Pais','Prioridade']).size().unstack(fill_value=0)

.groupby(['Pais','Prioridade']), vai fazer agrupação por categoria, assim agrupou 'Pais' e 'Prioridade'

.size(), mostra o tamanho de cada agrupamento feito

.unstack(fill_value=0), transforma o último nível do índice em colunas

In [12]:
paises_vendas_por_prioridade['Total'] = paises_vendas_por_prioridade.sum(axis=1)
paises_vendas_por_prioridade = paises_vendas_por_prioridade.sort_values('Total', ascending=False)
paises_vendas_por_prioridade.drop(columns='Total', inplace=True)
paises_vendas_por_prioridade = paises_vendas_por_prioridade[['Critico', 'Alto', 'Medio', 'Baixo']]

Cria uma coluna chamada 'Total'

.**sum**(**axis=1**)que mostra a contagem total de cada linha, axis=1(linha)

.**sort_values**('Total', **ascending=False**), faz o ordenamento decresente dosvalores da coluna total

.**drop**(columns='Total', **inplace=True**), Remove a coluna 'Total', inplace para atualizar o dataframe

 paises_vendas_por_prioridade[['Critico', 'Alto', 'Medio', 'Baixo']], Organiza a ordem das colunas

In [13]:
paises_vendas_por_prioridade.head(10)

Prioridade,Critico,Alto,Medio,Baixo
Pais,,,,
United States,783,3069,5709,432
Australia,221,902,1615,97
France,223,838,1624,142
Mexico,221,776,1509,138
Germany,168,542,1223,132
China,151,600,1034,95
United Kingdom,117,501,900,115
Brazil,122,486,936,55
India,132,437,936,50


Pergunta 4 - Qual foi a média de desconto nas vendas por subcategoria de produto?

In [14]:
media_vendas_subcategorias = dataset.groupby(['SubCategoria','Desconto']).size()
media_vendas_subcategorias

SubCategoria  Desconto
Accessories   0.00        1972
              0.10         174
              0.20         351
              0.40         198
              0.45          37
                          ... 
Tables        0.57          12
              0.60          28
              0.70          58
              0.80          12
              0.85           2
Length: 188, dtype: int64

In [15]:
dataset['SubCategoria'].unique()

<StringArray>
['Accessories',      'Chairs',      'Phones',     'Copiers',      'Tables',
     'Binders',    'Supplies',  'Appliances',    'Machines',   'Bookcases',
     'Storage', 'Furnishings',         'Art',       'Paper',   'Envelopes',
   'Fasteners',      'Labels']
Length: 17, dtype: str

In [16]:
valor_desconto = dataset.groupby('SubCategoria')['Desconto'].mean().reset_index()
valor_desconto.columns = ['SubCategoria', 'Media_Desconto']
valor_desconto.sort_values('Media_Desconto', ascending=False)

,SubCategoria,Media_Desconto
16,Tables,0.290732
3,Binders,0.179249
11,Machines,0.169583
5,Chairs,0.163087
4,Bookcases,0.153758
9,Furnishings,0.151066
13,Phones,0.145847
1,Appliances,0.141709
8,Fasteners,0.140595
14,Storage,0.138464


Pergunta 5 - Quais países tiveram maior média de valor de venda? Demonstre em um mapa.

In [17]:
media_de_vendas_por_pais = dataset.groupby('Pais')['Total_Vendas'].mean().reset_index().sort_values('Total_Vendas', ascending=False)
media_de_vendas_por_pais.columns = ['Pais', 'Média de Vendas']
media_de_vendas_por_pais.head(11)

,Pais,Média de Vendas
71,Lesotho,1118.665000
84,Montenegro,1001.092500
24,Chad,658.515000
126,Taiwan,546.259286
118,South Sudan,522.810000
114,Slovenia,489.980000
10,Bangladesh,480.101043
105,Republic of the Congo,452.205000
140,Uruguay,404.689333
65,Japan,403.150068
